In [97]:
import pandas as pd

df_train = pd.read_csv('data/train_transaction.csv')

df_train.shape

(590540, 394)

## Ordenar dataset por TransactionDT para luego separar el dataset en splits

In [98]:
df_train = df_train.sort_values('TransactionDT').reset_index(drop=True)

In [99]:
y = df_train['isFraud']
X = df_train.drop(columns=['isFraud'])

print(X.shape)
print(y.shape)

(590540, 393)
(590540,)


In [100]:
X.dtypes.value_counts()

float64    376
str         14
int64        3
Name: count, dtype: int64

In [101]:
X.select_dtypes(include='object').columns

C:\Users\juanc\AppData\Local\Temp\ipykernel_86604\2418744409.py:1: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  X.select_dtypes(include='object').columns


Index(['ProductCD', 'card4', 'card6', 'P_emaildomain', 'R_emaildomain', 'M1',
       'M2', 'M3', 'M4', 'M5', 'M6', 'M7', 'M8', 'M9'],
      dtype='str')

M1 a M9 — son columnas de match, sus valores son solo "T" o "F" (verdadero/falso)



In [102]:
X[['M1', 'M2', 'M3', 'M4', 'M5', 'M6', 'M7', 'M8', 'M9']].value_counts()

M1  M2  M3  M4  M5  M6  M7  M8  M9
T   T   T   M0  T   F   F   F   T     11558
                F   F   F   F   T     10064
                    T   F   F   T      8555
                T   T   F   F   T      7373
                F   F   F   F   F      5734
                                      ...  
F   T   F   M0  F   T   F   F   F         1
T   T   F   M1  F   F   T   T   F         1
        T   M0  T   F   F   T   F         1
    F   F   M0  F   T   T   F   T         1
    T   T   M0  T   T   F   T   F         1
Name: count, Length: 187, dtype: int64

In [103]:
X['M4'].value_counts()

M4
M0    196405
M2     59865
M1     52826
Name: count, dtype: int64

## Tipos de columnas categóricas

### Grupo M — columnas de match
- `M1, M2, M3, M5, M6, M7, M8, M9` → binarias (T/F) → mapeo simple: T=1, F=0
- `M4` → 3 valores (M0, M1, M2) → necesita encoding separado

### Grupo principal — categorías reales
- `ProductCD`, `card4`, `card6` → pocas categorías, valores de negocio
- `P_emaildomain`, `R_emaildomain` → dominios de email, muchos valores únicos posibles

In [104]:
X[['M1', 'M2', 'M3', 'M5', 'M6', 'M7', 'M8', 'M9']] = X[['M1', 'M2', 'M3', 'M5', 'M6', 'M7', 'M8', 'M9']].apply(lambda x: x.map({'T': 1, 'F': 0}))

In [105]:
X['M4'].value_counts()

M4
M0    196405
M2     59865
M1     52826
Name: count, dtype: int64

## Encoding de M4

`M4` tiene 3 valores: `M0`, `M1`, `M2`. Se aplica **ordinal encoding** (M0→0, M1→1, M2→2) por dos razones:

1. El nombre de los valores sugiere un índice implícito, Verizon los nombró con orden numérico.
2. One-hot encoding generaría columnas adicionales innecesarias para un modelo de árboles como LightGBM, que maneja ordinales bien sin perder información.

> Limitación: no sabemos si el orden real es M0 < M1 < M2. Si el modelo falla, revisar este encoding es un buen punto de partida.

In [106]:
X['M4'] = X['M4'].map(({'M0':0, 'M1':1, 'M2':2}))

In [107]:
X['M4'].value_counts()

M4
0.0    196405
2.0     59865
1.0     52826
Name: count, dtype: int64

## `.apply()` con lambda — comportamiento según el objeto

- `DataFrame.apply(lambda x: ...)` → `x` es una columna completa (Serie); `x.map()` funciona.
- `Serie.apply(lambda x: ...)` → `x` es un valor individual (string, int, etc.); `x.map()` no existe.
- No ovldiar, si voy a manejar columnas, las manejo como columnas/series, y si manejo texto, la manejo como texto antivo.

In [108]:
X[['ProductCD', 'card4', 'card6', 'P_emaildomain', 'R_emaildomain']].nunique()

ProductCD         5
card4             4
card6             4
P_emaildomain    59
R_emaildomain    60
dtype: int64

In [110]:
for col in ['ProductCD', 'card4', 'card6', 'P_emaildomain', 'R_emaildomain']:
    print(f"\n{col}:")
    print(X[col].value_counts())


ProductCD:
ProductCD
W    439670
C     68519
R     37699
H     33024
S     11628
Name: count, dtype: int64

card4:
card4
visa                384767
mastercard          189217
american express      8328
discover              6651
Name: count, dtype: int64

card6:
card6
debit              439938
credit             148986
debit or credit        30
charge card            15
Name: count, dtype: int64

P_emaildomain:
P_emaildomain
gmail.com           228355
yahoo.com           100934
hotmail.com          45250
anonymous.com        36998
aol.com              28289
comcast.net           7888
icloud.com            6267
outlook.com           5096
msn.com               4092
att.net               4033
live.com              3041
sbcglobal.net         2970
verizon.net           2705
ymail.com             2396
bellsouth.net         1909
yahoo.com.mx          1543
me.com                1522
cox.net               1393
optonline.net         1011
charter.net            816
live.com.mx            749
roc

In [111]:
X.select_dtypes(include='object').columns

C:\Users\juanc\AppData\Local\Temp\ipykernel_86604\2418744409.py:1: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  X.select_dtypes(include='object').columns


Index(['ProductCD', 'card4', 'card6', 'P_emaildomain', 'R_emaildomain'], dtype='str')

## Split temporal vs. aleatorio

En datos financieros con componente de tiempo, el split debe ser **cronológico**, no aleatorio.

- Split aleatorio → mezcla datos futuros en el entrenamiento → el modelo aprende patrones que en producción no existirían aún → métricas infladas artificialmente (**temporal leakage**).
- Split cronológico → entrena con el pasado, valida con el futuro → simula exactamente el escenario real de producción.

> Regla: siempre predices el futuro con datos del pasado. El split debe reflejar eso.

## Data leakage — regla general

El modelo nunca debe ver información que no tendría disponible en producción al momento de predecir.

Dos formas comunes:
- **Target leakage:** usar la variable objetivo (o derivados de ella) para calcular transformaciones antes de separar train/validation.
- **Temporal leakage:** mezclar datos futuros en el entrenamiento cuando los datos tienen componente de tiempo.

> Solución: hacer el split **antes** de calcular cualquier encoding o transformación que use estadísticas del dataset.

In [ ]:
%pip install scikit-learn

## **Dividir dataset**

Se divide el dataset teniendo ordenado todo por tiempo de la variable TransactionDT, para tener el contexto real de como se decta, no de manera aleatoria

In [112]:
X = X.sort_values('TransactionDT')

corte = int(len(X)*0.8)

X_train = X.iloc[:corte]
y_train = y.iloc[:corte]

X_Val = X.iloc[corte:]
y_Val = y.iloc[corte:]

In [113]:
X_train.shape, X_Val.shape, y_train.shape, y_Val.shape

((472432, 393), (118108, 393), (472432,), (118108,))

## Decisión técnica: Target Encoding después del split

**Técnica elegida:** Target encoding para P_emaildomain y R_emaildomain  
**Calculado sobre:** X_train / y_train únicamente — luego aplicado a X_val

**Por qué después del split:**  
El target encoding reemplaza cada categoría con la tasa de fraude promedio de esa categoría.  
Si se calcula antes del split (sobre todo el dataset), las etiquetas de validación contaminan  
el cálculo — data leakage. El modelo "ve el futuro" y el AUC en validación queda inflado,  
no refleja rendimiento real.

**Supuesto que asume:**  
La distribución de dominios de correo en producción será similar a la de train.  
Dominios nuevos (no vistos en train) recibirán el promedio global como fallback.

## **Encoding a target encoding a p_emaildomain**

In [114]:
tasa_fraude_dominio = y_train.groupby(X_train['P_emaildomain']).mean()

print(tasa_fraude_dominio)

P_emaildomain
aim.com             0.149254
anonymous.com       0.023264
aol.com             0.021885
att.net             0.006384
bellsouth.net       0.028481
cableone.net        0.024590
centurylink.net     0.000000
cfl.rr.com          0.000000
charter.net         0.023256
comcast.net         0.030018
cox.net             0.021409
earthlink.net       0.014706
embarqmail.com      0.033333
frontier.com        0.004525
frontiernet.net     0.032680
gmail               0.019465
gmail.com           0.044003
gmx.de              0.000000
hotmail.co.uk       0.000000
hotmail.com         0.052021
hotmail.de          0.000000
hotmail.es          0.066946
hotmail.fr          0.000000
icloud.com          0.031803
juno.com            0.022140
live.com            0.031919
live.com.mx         0.028846
live.fr             0.000000
mac.com             0.025000
mail.com            0.194690
me.com              0.013396
msn.com             0.017665
netzero.com         0.000000
netzero.net         0.006369


## **protonmail cuenta con el porcentaje de fraude más alto, con un 46%**

In [115]:
X_train['P_emaildomain'] = X_train['P_emaildomain'].map(tasa_fraude_dominio)

In [116]:
X_Val['P_emaildomain'] = X_Val['P_emaildomain'].map(tasa_fraude_dominio).fillna(y_train.mean())

## **Encoding a target encoding a R_emaildomain**

In [117]:
tasa_fraude_R_emaildomain = y_train.groupby(X_train['R_emaildomain']).mean()
print(tasa_fraude_R_emaildomain)

R_emaildomain
aim.com             0.033333
anonymous.com       0.028545
aol.com             0.036878
att.net             0.000000
bellsouth.net       0.000000
cableone.net        0.000000
centurylink.net     0.000000
cfl.rr.com          0.000000
charter.net         0.045455
comcast.net         0.013308
cox.net             0.014963
earthlink.net       0.088235
embarqmail.com      0.000000
frontier.com        0.000000
frontiernet.net     0.000000
gmail               0.000000
gmail.com           0.115162
gmx.de              0.000000
hotmail.co.uk       0.000000
hotmail.com         0.074530
hotmail.de          0.000000
hotmail.es          0.068966
hotmail.fr          0.000000
icloud.com          0.118629
juno.com            0.000000
live.com            0.046062
live.com.mx         0.028939
live.fr             0.000000
mac.com             0.000000
mail.com            0.360360
me.com              0.002105
msn.com             0.001393
netzero.com         0.000000
netzero.net         0.125000


In [118]:
X_train['R_emaildomain'] = X_train['R_emaildomain'].map(tasa_fraude_R_emaildomain)

In [119]:
X_Val['R_emaildomain'] = X_Val['R_emaildomain'].map(tasa_fraude_R_emaildomain).fillna(y_train.mean())

**One_Hot_Encoding** campor de productCD, card4 y card6

In [124]:
from sklearn.preprocessing import OneHotEncoder

encoder = OneHotEncoder(sparse_output=False, handle_unknown='ignore')

X_train_encoded = encoder.fit_transform(X_train[['ProductCD', 'card4', 'card6']])

X_Val_encoded = encoder.transform(X_Val[['ProductCD', 'card4', 'card6']])

X_train.drop(columns=['ProductCD', 'card4', 'card6'], inplace=True)
X_Val.drop(columns=['ProductCD', 'card4', 'card6'], inplace=True)



In [125]:
X_train = pd.concat([X_train.reset_index(drop=True), pd.DataFrame(X_train_encoded, columns=encoder.get_feature_names_out())], axis=1)
X_Val = pd.concat([X_Val.reset_index(drop=True), pd.DataFrame(X_Val_encoded, columns=encoder.get_feature_names_out())], axis=1)

In [126]:
X_train.shape, X_Val.shape, y_train.shape, y_Val.shape

((472432, 405), (118108, 405), (472432,), (118108,))

In [127]:
X_train.select_dtypes(include='object').columns

Index([], dtype='str')

**Guardar dataset en formato parquet**

In [133]:
%pip install fastparquet

  Using cached fsspec-2026.4.0-py3-none-any.whl.metadata (10 kB)
   ---------------------------------------- 0.0/701.7 kB ? eta -:--:--
   ---------------------------------------- 701.7/701.7 kB 15.7 MB/s  0:00:00
   ---------------------------------------- 0.0/1.7 MB ? eta -:--:--
   ---------------------------------------- 1.7/1.7 MB 21.3 MB/s  0:00:00
Using cached fsspec-2026.4.0-py3-none-any.whl (203 kB)

   ---------------------------------------- 0/3 [fsspec]
   ---------------------------------------- 0/3 [fsspec]
   ---------------------------------------- 0/3 [fsspec]
   ---------------------------------------- 0/3 [fsspec]
   ---------------------------------------- 0/3 [fsspec]
   -------------------------- ------------- 2/3 [fastparquet]
   -------------------------- ------------- 2/3 [fastparquet]
   ---------------------------------------- 3/3 [fastparquet]

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.0.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [135]:
import fastparquet 

X_train.to_parquet('data/X_train.parquet', engine='fastparquet')
y_train.to_frame().to_parquet('data/y_train.parquet', engine='fastparquet')
X_Val.to_parquet('data/X_Val.parquet', engine='fastparquet')
y_Val.to_frame().to_parquet('data/y_Val.parquet', engine='fastparquet')